# STEP 0 — Setup & One-Time Model Cache

**Run this ONCE. Never again.**

What it does:
1. Verifies both T4s are available
2. Downloads all 4 models into `/kaggle/working/hf_cache/`
3. Zips the cache → you upload it as the `slm-agents-models` Kaggle Dataset

After this step, every future session loads models in seconds from the cached dataset instead of re-downloading ~20GB.

**Datasets to create on Kaggle before running any other notebook:**
- `slm-agents-code` — upload the `src/` folder from this project
- `slm-agents-results` — create empty, will be updated after each step
- `slm-agents-models` — created from the output of THIS notebook

In [ ]:
!pip install -q transformers==4.44.0 accelerate bitsandbytes datasets
!pip install -q rank_bm25 scikit-learn pandas numpy matplotlib seaborn
print('✓ packages installed')

In [ ]:
import subprocess, torch
from pathlib import Path

result = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.free','--format=csv,noheader'], capture_output=True, text=True)
print('GPUs:', result.stdout)
print(f'CUDA available: {torch.cuda.is_available()}')
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')

In [ ]:
import os
HF_CACHE = Path('/kaggle/working/hf_cache')
HF_CACHE.mkdir(exist_ok=True)
os.environ['HF_HOME']            = str(HF_CACHE)
os.environ['TRANSFORMERS_CACHE'] = str(HF_CACHE / 'hub')
os.environ['HF_DATASETS_CACHE']  = str(HF_CACHE / 'datasets')
print(f'HF cache → {HF_CACHE}')

In [ ]:
# Download all 4 models — this takes ~20 min total
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import gc

MODELS = [
    ('Qwen/Qwen2.5-1.5B-Instruct',              False),
    ('Qwen/Qwen2.5-3B-Instruct',                False),
    ('microsoft/Phi-3-mini-4k-instruct',         False),
    ('meta-llama/Meta-Llama-3.1-8B-Instruct',    True),  # 4-bit
]

for hf_id, use_4bit in MODELS:
    print(f'\n── Downloading {hf_id} ──────────────')
    tok = AutoTokenizer.from_pretrained(hf_id, trust_remote_code=True)
    if use_4bit:
        bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype='float16', bnb_4bit_use_double_quant=True, bnb_4bit_quant_type='nf4')
        m   = AutoModelForCausalLM.from_pretrained(hf_id, quantization_config=bnb, device_map='auto', trust_remote_code=True)
    else:
        m   = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype='float16', device_map='auto', trust_remote_code=True)
    print(f'  ✓ {hf_id}')
    del m, tok
    gc.collect()
    import torch; torch.cuda.empty_cache()

print('\n✅ All models downloaded to HF cache')

In [ ]:
# Smoke test — load smallest model and run one inference
import sys
sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.models import get_model_by_name, load_model
from src.inference import generate_single_with_logits, build_react_prompt
from src.react_env import build_react_prompt

config = get_model_by_name('Qwen2.5-1.5B')
model, tokenizer = load_model(config)

prompt = build_react_prompt('What is the capital of France?', '')
out = generate_single_with_logits(model, tokenizer, prompt, max_new_tokens=64)
print('Response:', out['response'])
print('Entropy:', out['first_token_entropy'])
print('Top1 prob:', out['top1_prob'])
print('\n✅ Smoke test passed')

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Zip model cache for upload as slm-agents-models dataset
import sys
sys.path.insert(0, '/kaggle/working/slm_agents_paper')
from src.session_save import save_model_cache
save_model_cache(HF_CACHE)
print('\n✅ STEP 0 COMPLETE')
print('   Download hf_cache_upload.zip from output panel')
print('   Upload as new Kaggle Dataset: slm-agents-models')